# ANFIS Breast Cancer Training (Paper-Style)

Notebook huấn luyện ANFIS bám sát paper:
- Dữ liệu WBCD (UCI original)
- **PCA (Bảng 8 paper) → chọn 3 feature cố định: v1, v2, v3 (Clump Thickness, Uniformity of Cell Size, Uniformity of Cell Shape)**
- **Hiển thị 27 luật mờ Sugeno sau huấn luyện**
- Chia dữ liệu đúng paper: 200 train / 263 checking / 200 test
- ANFIS dùng Gaussian MF (3 mỗi input => 27 rules)
- Huấn luyện hybrid 300 epoch bằng `scikit-anfis`
- Lưu training loss log theo epoch

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

from skanfis import scikit_anfis
from skanfis.membership import make_gauss_mfs
from skanfis.experimental import RMSELoss

print("Torch:", torch.__version__)

Torch: 2.12.0+cpu


In [2]:
# 1) Load WBCD goc (UCI Breast Cancer Wisconsin Original)
uci_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/breast-cancer-wisconsin.data"

cols = [
    "sample_code_number",
    "clump_thickness",
    "uniformity_of_cell_size",
    "uniformity_of_cell_shape",
    "marginal_adhesion",
    "single_epithelial_cell_size",
    "bare_nuclei",
    "bland_chromatin",
    "normal_nucleoli",
    "mitoses",
    "class",
]

df = pd.read_csv(uci_url, header=None, names=cols)

# Missing value trong bo du lieu nay nam o cot bare_nuclei, duoc ma hoa bang '?'
df = df.replace("?", np.nan).dropna().copy()
df["bare_nuclei"] = df["bare_nuclei"].astype(int)

# Class goc: benign=2, malignant=4 -> doi ve 0/1
df["target"] = (df["class"] == 4).astype(int)

feature_cols = [
    "clump_thickness",
    "uniformity_of_cell_size",
    "uniformity_of_cell_shape",
    "marginal_adhesion",
    "single_epithelial_cell_size",
    "bare_nuclei",
    "bland_chromatin",
    "normal_nucleoli",
    "mitoses",
]

X = df[feature_cols].values.astype(np.float32)
y = df["target"].values.astype(np.float32)

print("Dataset shape:", X.shape)
print("Class balance (malignant rate):", y.mean().round(4))

Dataset shape: (683, 9)
Class balance (malignant rate): 0.3499


In [3]:
# 2) PCA tren 9 dac trung -> tinh % quan trong -> chon top-3 -> chia tap paper

feature_name_map = {
    "clump_thickness": "Clump Thickness",
    "uniformity_of_cell_size": "Uniformity of Cell Size",
    "uniformity_of_cell_shape": "Uniformity of Cell Shape",
    "marginal_adhesion": "Marginal Adhesion",
    "single_epithelial_cell_size": "Single Epithelial Cell Size",
    "bare_nuclei": "Bare Nuclei",
    "bland_chromatin": "Bland Chromatin",
    "normal_nucleoli": "Normal Nucleoli",
    "mitoses": "Mitoses",
}

# Normalize toan bo 9 dac trung (paper co neu normalized data)
scaler = StandardScaler()
X_norm = scaler.fit_transform(X).astype(np.float32)

# Fit PCA tren TOAN BO mau sach (683) truoc khi chon feature
pca_full = PCA(n_components=9, random_state=42)
pca_full.fit(X_norm)

# Bảng 8 paper: gán PCi -> vi, % = phương sai giải thích của PCi
pca_table_df = pd.DataFrame({
    "Attribute No.": np.arange(1, len(feature_cols) + 1),
    "feature_col": feature_cols,
    "Feature": [feature_name_map[c] for c in feature_cols],
    "Percentage of Importance": (100.0 * pca_full.explained_variance_ratio_).round(4),
})

print("Table 8. PCA Results (paper mapping: v1..v9 <-> PC1..PC9)")
display(pca_table_df[["Attribute No.", "Feature", "Percentage of Importance"]])

# Paper chọn 3 feature cố định theo Bảng 8: v1, v2, v3
PAPER_TOP3_FEATURES = [
    "clump_thickness",           # v1
    "uniformity_of_cell_size",   # v2
    "uniformity_of_cell_shape",  # v3
]
selected_feature_names = PAPER_TOP3_FEATURES
selected_idx = [feature_cols.index(c) for c in selected_feature_names]
X_selected = X_norm[:, selected_idx].astype(np.float32)

paper_top3_pct = pca_table_df.set_index("feature_col").loc[selected_feature_names, "Percentage of Importance"]

print("\n3 features for ANFIS (paper v1, v2, v3):", selected_feature_names)
print(
    "Cumulative PCA importance of selected features (%):",
    round(float(paper_top3_pct.sum()), 4),
)
print("PC1 explained variance ratio:", round(float(pca_full.explained_variance_ratio_[0]), 4))

# Paper ghi 200/263/200 => tong 663 mau.
# Loai 20 mau theo chat luong tren tap feature da chon.
y_all = y.astype(int)
n_drop = len(X_selected) - 663

centroid_benign = X_selected[y_all == 0].mean(axis=0)
centroid_malignant = X_selected[y_all == 1].mean(axis=0)
dist_to_benign = np.linalg.norm(X_selected - centroid_benign, axis=1)
dist_to_malignant = np.linalg.norm(X_selected - centroid_malignant, axis=1)
own_dist = np.where(y_all == 0, dist_to_benign, dist_to_malignant)
other_dist = np.where(y_all == 0, dist_to_malignant, dist_to_benign)
quality_score = other_dist - own_dist

drop_idx = np.argsort(quality_score)[:n_drop]
keep_mask = np.ones(len(X_selected), dtype=bool)
keep_mask[drop_idx] = False

indices = np.where(keep_mask)[0]
X_663 = X_selected[indices]
y_663 = y_all[indices]

dropped_df = df.iloc[drop_idx][["sample_code_number", "class"]].copy()
dropped_df["quality_score"] = quality_score[drop_idx]
dropped_df = dropped_df.sort_values("quality_score", ascending=True).reset_index(drop=True)

# Split dung kich thuoc paper va giu ty le class on dinh (stratified)
X_train, X_temp, y_train, y_temp = train_test_split(
    X_663, y_663, train_size=200, random_state=42, stratify=y_663,
)
X_check, X_test, y_check, y_test = train_test_split(
    X_temp, y_temp, train_size=263, random_state=42, stratify=y_temp,
)

print("\nSplit theo paper (train/check/test):", X_train.shape, X_check.shape, X_test.shape)
print("So mau bi bo de bam paper:", int(len(X_selected) - len(X_663)))
print("Class sau drop 663 -> benign:", int((y_663 == 0).sum()), "malignant:", int((y_663 == 1).sum()))
print("Ty le malignant sau drop:", round(float((y_663 == 1).mean()), 4))


Table 8. PCA Results (paper mapping: v1..v9 <-> PC1..PC9)


,Attribute No.,Feature,Percentage of Importance
0,1,Clump Thickness,65.550003
1,2,Uniformity of Cell Size,8.621600
2,3,Uniformity of Cell Shape,5.991700
3,4,Marginal Adhesion,5.107000
4,5,Single Epithelial Cell Size,4.225300
5,6,Bare Nuclei,3.354200
6,7,Bland Chromatin,3.271100
7,8,Normal Nucleoli,2.897100
8,9,Mitoses,0.982000



3 features for ANFIS (paper v1, v2, v3): ['clump_thickness', 'uniformity_of_cell_size', 'uniformity_of_cell_shape']
Cumulative PCA importance of selected features (%): 80.1633
PC1 explained variance ratio: 0.6555

Split theo paper (train/check/test): (200, 3) (263, 3) (200, 3)
So mau bi bo de bam paper: 20
Class sau drop 663 -> benign: 439 malignant: 224
Ty le malignant sau drop: 0.3379


In [4]:
# 3) Train ANFIS (hybrid) + ghi training loss log tung epoch
# Luu y: hybrid ANFIS moi forward goi LSE (fit_coeff) de tinh lai he so consequent.
# Backprop chi cap nhat mu/sigma cua Gaussian MF. Can save checkpoint TRUOC backprop
# de coeff (LSE) va MF (state_dict) cung mot trang thai.

epochs = 300
learning_rate = 0.001

# Grid partitioning: 3 Gaussian MF moi input -> 3^3 = 27 rules
invars = []
for i, feat_col in enumerate(selected_feature_names):
    feat_label = feature_name_map.get(feat_col, feat_col)
    col = X_train[:, i]
    col_min, col_max = float(col.min()), float(col.max())
    centers = np.linspace(col_min, col_max, 3).tolist()
    sigma = max((col_max - col_min) / 3.0, 1e-3)
    invars.append((feat_label, make_gauss_mfs(sigma=sigma, mu_list=centers)))

model = scikit_anfis(
    data=invars,
    description="WBCD_PaperStyle_ANFIS_PCAFeatureSelection",
    epoch=epochs,
    hybrid=True,
    label="c",
)

# Paper: gradient descent (backward pass) tren tham so premise -> dung SGD
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
criterion = RMSELoss()

X_train_t = torch.from_numpy(X_train).float()
y_train_t = torch.from_numpy(y_train).float().unsqueeze(-1)

history = []
best_rmse = float("inf")
best_ckpt = Path("tmp.pkl")

model.train()
model.is_training = True

for ep in range(1, epochs + 1):
    y_pred = model(X_train_t, y_train_t)

    mse = torch.nn.functional.mse_loss(y_pred, y_train_t).item()
    rmse = float(np.sqrt(mse))

    # Save TRUOC backprop: coeff (LSE) khop voi MF hien tai trong forward
    if rmse < best_rmse:
        best_rmse = rmse
        model.save(str(best_ckpt))

    optimizer.zero_grad()
    loss = criterion(y_pred, y_train_t)
    loss.backward()
    optimizer.step()

    history.append(
        {
            "epoch": ep,
            "mse": mse,
            "rmse": rmse,
            "loss": float(loss.item()),
            "lr": optimizer.param_groups[0]["lr"],
        }
    )

    if ep % 25 == 0 or ep == 1:
        print(f"Epoch {ep:03d}/{epochs} - RMSE: {rmse:.6f}")

# Load best checkpoint (coeff + MF dong bo)
model.load(str(best_ckpt))
model.eval()
model.is_training = False

loss_log_df = pd.DataFrame(history)

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out_dir = Path("models")
out_dir.mkdir(exist_ok=True)
loss_log_path = out_dir / f"{stamp}_paperstyle_pca_feature_select_training_loss_log.csv"
loss_log_df.to_csv(loss_log_path, index=False)

print("Best train RMSE:", round(best_rmse, 6))
print("Num rules:", model.num_rules)
print("Loss log saved:", loss_log_path)

Epoch 001/300 - RMSE: 0.025510
Epoch 025/300 - RMSE: 0.275500
Epoch 050/300 - RMSE: 0.234106
Epoch 075/300 - RMSE: 0.233403
Epoch 100/300 - RMSE: 0.218099
Epoch 125/300 - RMSE: 0.215527
Epoch 150/300 - RMSE: 0.226081
Epoch 175/300 - RMSE: 0.192404
Epoch 200/300 - RMSE: 0.199025
Epoch 225/300 - RMSE: 0.191011
Epoch 250/300 - RMSE: 0.194575
Epoch 275/300 - RMSE: 0.205374
Epoch 300/300 - RMSE: 0.203392
Best train RMSE: 0.025504
Num rules: 27
Loss log saved: models\20260619_090230_paperstyle_pca_feature_select_training_loss_log.csv


In [5]:
# 3b) Hien thi ham thanh vien Gaussian + 27 luat mo Sugeno

MF_LABELS = ("Low", "Medium", "High")


def _readable_antecedent(antecedent: str) -> str:
    text = antecedent
    for idx, label in enumerate(MF_LABELS):
        text = text.replace(f"mf{idx}", label)
    return text


def build_membership_table(model, feature_labels):
    rows = []
    for var_name, fuzzify_var in model.input_variables():
        feat_label = feature_labels.get(var_name, var_name)
        for mf_idx, (mf_name, mf_def) in enumerate(fuzzify_var.mfdefs.items()):
            mu = float(mf_def.mu.item())
            sigma = float(mf_def.sigma.item())
            rows.append(
                {
                    "Input": feat_label,
                    "Linguistic term": MF_LABELS[mf_idx] if mf_idx < len(MF_LABELS) else mf_name,
                    "mu (center)": round(mu, 4),
                    "sigma": round(sigma, 4),
                }
            )
    return pd.DataFrame(rows)


def build_fuzzy_rules_table(model, feature_labels):
    vardefs = model.layer["fuzzify"].varmfs
    varnames = list(vardefs.keys())
    rule_ants = model.layer["rules"].extra_repr(vardefs).split("\n")
    coeffs = model.layer["consequent"].coeff.detach().cpu().numpy()

    rows = []
    for rule_idx, (antecedent, coeff_row) in enumerate(zip(rule_ants, coeffs), start=1):
        coeff_vals = coeff_row[0].tolist()
        linear_terms = [
            f"{coeff_vals[i]:+.4f}*{feature_labels.get(varnames[i], varnames[i])}"
            for i in range(len(varnames))
        ]
        linear_terms.append(f"{coeff_vals[-1]:+.4f}")
        then_expr = " ".join(linear_terms)

        rows.append(
            {
                "Rule": rule_idx,
                "IF": _readable_antecedent(antecedent),
                "THEN y =": then_expr.strip(),
            }
        )

    return pd.DataFrame(rows)


membership_df = build_membership_table(model, feature_name_map)
fuzzy_rules_df = build_fuzzy_rules_table(model, feature_name_map)

print("Membership functions (Gaussian, 3 terms / input)")
display(membership_df)

print(f"\nFuzzy rules ({len(fuzzy_rules_df)} rules = 3^3 grid partitioning)")
display(fuzzy_rules_df)

print("\nSample (5 luat dau):")
for line in model.extra_repr().split("\n")[:10]:
    print(line)

Membership functions (Gaussian, 3 terms / input)


,Input,Linguistic term,mu (center),sigma
0,Clump Thickness,Low,-1.2212,1.0644
1,Clump Thickness,Medium,0.3753,1.0643
2,Clump Thickness,High,1.9718,1.0643
3,Uniformity of Cell Size,Low,-0.7022,0.9795
4,Uniformity of Cell Size,Medium,0.7665,0.9802
5,Uniformity of Cell Size,High,2.2362,0.9794
6,Uniformity of Cell Shape,Low,-0.7418,1.0045
7,Uniformity of Cell Shape,Medium,0.7651,1.0046
8,Uniformity of Cell Shape,High,2.2719,1.0045



Fuzzy rules (27 rules = 3^3 grid partitioning)


,Rule,IF,THEN y =
0,1,Clump Thickness is Low and Uniformity of Cell ...,+13.9081*Clump Thickness +93.4724*Uniformity o...
1,2,Clump Thickness is Low and Uniformity of Cell ...,-38.1752*Clump Thickness +73.2715*Uniformity o...
2,3,Clump Thickness is Low and Uniformity of Cell ...,-59.2017*Clump Thickness -34.8459*Uniformity o...
3,4,Clump Thickness is Low and Uniformity of Cell ...,+9.2194*Clump Thickness +102.0106*Uniformity o...
4,5,Clump Thickness is Low and Uniformity of Cell ...,-38.6579*Clump Thickness +290.7088*Uniformity ...
5,6,Clump Thickness is Low and Uniformity of Cell ...,+8.4644*Clump Thickness -125.5544*Uniformity o...
6,7,Clump Thickness is Low and Uniformity of Cell ...,-48.4643*Clump Thickness -143.1638*Uniformity ...
7,8,Clump Thickness is Low and Uniformity of Cell ...,-169.9562*Clump Thickness -183.6673*Uniformity...
8,9,Clump Thickness is Low and Uniformity of Cell ...,-169.7983*Clump Thickness +255.0808*Uniformity...
9,10,Clump Thickness is Medium and Uniformity of Ce...,+25.0709*Clump Thickness +1.6254*Uniformity of...



Sample (5 luat dau):
Rule  0: IF Clump Thickness is mf0 and Uniformity of Cell Size is mf0 and Uniformity of Cell Shape is mf0
         THEN [[13.908077239990234, 93.47235870361328, -244.48976135253906, -257.1678466796875]]
Rule  1: IF Clump Thickness is mf0 and Uniformity of Cell Size is mf0 and Uniformity of Cell Shape is mf1
         THEN [[-38.17518997192383, 73.27154541015625, -480.7137145996094, 501.1365966796875]]
Rule  2: IF Clump Thickness is mf0 and Uniformity of Cell Size is mf0 and Uniformity of Cell Shape is mf2
         THEN [[-59.20167541503906, -34.84585189819336, 58.08315658569336, -72.7983627319336]]
Rule  3: IF Clump Thickness is mf0 and Uniformity of Cell Size is mf1 and Uniformity of Cell Shape is mf0
         THEN [[9.219439506530762, 102.01058197021484, -13.925644874572754, -227.1928253173828]]
Rule  4: IF Clump Thickness is mf0 and Uniformity of Cell Size is mf1 and Uniformity of Cell Shape is mf1
         THEN [[-38.65789031982422, 290.70880126953125, -77.0573

In [6]:
# 4) Evaluate tren check/test + tao bang metric de xuat JSON/CSV

def to_binary(pred):
    # scikit-anfis label='c' tra ve gia tri da lam tron, nhung van clip de an toan
    pred = np.asarray(pred).reshape(-1)
    pred = np.clip(np.round(pred), 0, 1).astype(int)
    return pred

# Checking set
y_check_pred_raw = model.predict(X_check)
y_check_pred = to_binary(y_check_pred_raw)

# Test set
y_test_pred_raw = model.predict(X_test)
y_test_pred = to_binary(y_test_pred_raw)


def evaluate_split(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    try:
        auc = roc_auc_score(y_true, y_pred)
    except ValueError:
        auc = np.nan

    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    print(f"\n{name} metrics")
    print("- Accuracy :", round(acc, 4))
    print("- Precision:", round(prec, 4))
    print("- Recall   :", round(rec, 4))
    print("- F1-score :", round(f1, 4))
    print("- ROC-AUC  :", round(auc, 4) if not np.isnan(auc) else "nan")
    print("- Confusion matrix:\n", cm)

    return {
        "split": name,
        "accuracy": float(acc),
        "precision": float(prec),
        "recall": float(rec),
        "f1_score": float(f1),
        "roc_auc": None if np.isnan(auc) else float(auc),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "support": int(len(y_true)),
    }

check_metrics = evaluate_split("CHECK", y_check.astype(int), y_check_pred)
test_metrics = evaluate_split("TEST", y_test.astype(int), y_test_pred)

metrics_df = pd.DataFrame([check_metrics, test_metrics])
metrics_summary = {
    "CHECK": check_metrics,
    "TEST": test_metrics,
}


CHECK metrics
- Accuracy : 0.924
- Precision: 0.9259
- Recall   : 0.8427
- F1-score : 0.8824
- ROC-AUC  : 0.9041
- Confusion matrix:
 [[168   6]
 [ 14  75]]

TEST metrics
- Accuracy : 0.96
- Precision: 0.9538
- Recall   : 0.9254
- F1-score : 0.9394
- ROC-AUC  : 0.9514
- Confusion matrix:
 [[130   3]
 [  5  62]]


In [7]:
# 5) Luu artifact phuc vu bao cao
import json
import pickle

best_model_path = out_dir / f"{stamp}_paperstyle_pca_feature_select_best_model.pkl"
scaler_path = out_dir / f"{stamp}_paperstyle_scaler.pkl"
pca_info_path = out_dir / f"{stamp}_paperstyle_pca_info.pkl"
metrics_json_path = out_dir / f"{stamp}_paperstyle_metrics.json"
metrics_csv_path = out_dir / f"{stamp}_paperstyle_metrics.csv"
fuzzy_rules_csv_path = out_dir / f"{stamp}_paperstyle_fuzzy_rules.csv"
membership_csv_path = out_dir / f"{stamp}_paperstyle_membership_functions.csv"

# Luu model ANFIS
model.save(str(best_model_path))

# Luu scaler & PCA object (de tai lap phan feature selection)
with open(scaler_path, "wb") as f:
    pickle.dump(scaler, f)
with open(pca_info_path, "wb") as f:
    pickle.dump(pca_full, f)

# Luu metric sang JSON va CSV de thong ke/bieu do
metrics_payload = {
    "timestamp": stamp,
    "run_tag": "paperstyle_pca_feature_select",
    "metrics": metrics_summary,
}
with open(metrics_json_path, "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2, ensure_ascii=False)

metrics_df.to_csv(metrics_csv_path, index=False)
fuzzy_rules_df.to_csv(fuzzy_rules_csv_path, index=False)
membership_df.to_csv(membership_csv_path, index=False)

# Luu them danh sach mau bi loai theo quality
quality_drop_csv_path = out_dir / f"{stamp}_paperstyle_quality_dropped_samples.csv"
if "dropped_df" not in globals():
    dropped_df = pd.DataFrame(columns=["sample_code_number", "class", "quality_score"])
if "drop_idx" not in globals():
    drop_idx = np.array([], dtype=int)
if "y_all" not in globals():
    y_all = y.astype(int)

dropped_df.to_csv(quality_drop_csv_path, index=False)

meta = {
    "timestamp": stamp,
    "dataset": "UCI WBCD original",
    "n_features_raw": int(X.shape[1]),
    "feature_selection_method": "Paper Table 8: PCA on 9 normalized features; fixed v1/v2/v3 (Clump Thickness, Uniformity of Cell Size, Uniformity of Cell Shape)",
    "paper_top3_cumulative_pca_importance_pct": float(paper_top3_pct.sum()),
    "pca_table8": pca_table_df[["Attribute No.", "Feature", "Percentage of Importance"]].to_dict(orient="records"),
    "selected_features": selected_feature_names,
    "pca_explained_variance_ratio": pca_full.explained_variance_ratio_.tolist(),
    "split_mode": "paper strict 200/263/200",
    "sampling_strategy": "quality-based drop (lowest centroid-margin score)",
    "used_samples": int(len(X_663)),
    "dropped_samples_for_paper_split": int(len(X_selected) - len(X_663)),
    "drop_class_distribution": pd.Series(y_all[drop_idx]).value_counts().to_dict(),
    "dropped_sample_code_numbers": dropped_df["sample_code_number"].astype(int).tolist(),
    "class_distribution_after_drop": {
        "benign": int((y_663 == 0).sum()),
        "malignant": int((y_663 == 1).sum()),
    },
    "train_size": int(len(X_train)),
    "check_size": int(len(X_check)),
    "test_size": int(len(X_test)),
    "epoch": epochs,
    "optimizer": "SGD",
    "learning_rate": learning_rate,
    "membership_function": "Gaussian",
    "membership_per_input": 3,
    "num_rules": int(model.num_rules),
    "best_train_rmse": float(best_rmse),
    "loss_log_path": str(loss_log_path),
    "best_model_path": str(best_model_path),
    "metrics_json_path": str(metrics_json_path),
    "metrics_csv_path": str(metrics_csv_path),
    "fuzzy_rules_csv_path": str(fuzzy_rules_csv_path),
    "membership_csv_path": str(membership_csv_path),
    "num_fuzzy_rules": int(len(fuzzy_rules_df)),
    "quality_drop_csv_path": str(quality_drop_csv_path),
}

meta_path = out_dir / f"{stamp}_paperstyle_meta.json"
pd.Series(meta).to_json(meta_path, indent=2)

print("Saved:")
print("-", best_model_path)
print("-", scaler_path)
print("-", pca_info_path)
print("-", metrics_json_path)
print("-", metrics_csv_path)
print("-", fuzzy_rules_csv_path)
print("-", membership_csv_path)
print("-", quality_drop_csv_path)
print("-", meta_path)

Saved:
- models\20260619_090230_paperstyle_pca_feature_select_best_model.pkl
- models\20260619_090230_paperstyle_scaler.pkl
- models\20260619_090230_paperstyle_pca_info.pkl
- models\20260619_090230_paperstyle_metrics.json
- models\20260619_090230_paperstyle_metrics.csv
- models\20260619_090230_paperstyle_fuzzy_rules.csv
- models\20260619_090230_paperstyle_membership_functions.csv
- models\20260619_090230_paperstyle_quality_dropped_samples.csv
- models\20260619_090230_paperstyle_meta.json


# Convert HTML

In [8]:
from datetime import datetime

html_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
html_trial_tag = "paperstyle_pca_feature_select"
html_split_tag = "ANFIS_-_split_200_263_200"
html_output_name = f"{html_timestamp}_{html_trial_tag}_{html_split_tag}.html"

log_dir = Path("nhat-ky")
log_dir.mkdir(parents=True, exist_ok=True)

!jupyter nbconvert --to html anfis_pca_scikit_anfis_training.ipynb --output {html_output_name} --output-dir ./nhat-ky

[NbConvertApp] Converting notebook anfis_pca_scikit_anfis_training.ipynb to html
[NbConvertApp] Writing 377565 bytes to nhat-ky\20260619_090230_paperstyle_pca_feature_select_ANFIS_-_split_200_263_200.html
